In [ ]:
# Install once if required
# %pip install gensim nltk numpy matplotlib scikit-learn

import re
import numpy as np
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import gutenberg, stopwords
from nltk.tokenize import sent_tokenize, word_tokenize
from gensim.models import Word2Vec
from sklearn.decomposition import PCA

# Download required NLTK data
for package in ["gutenberg", "punkt", "punkt_tab", "stopwords"]:
    nltk.download(package, quiet=True)

# Load text dataset
raw_text = gutenberg.raw("carroll-alice.txt")
sentences = sent_tokenize(raw_text)

# Text preprocessing
stop_words = set(stopwords.words("english"))
data = []

for sentence in sentences:
    sentence = re.sub(r"[^A-Za-z\s]", " ", sentence).lower()
    words = [
        word for word in word_tokenize(sentence)
        if word not in stop_words and len(word) > 1
    ]
    if words:
        data.append(words)

print("Cleaned sentences:", len(data))
print("Sample:", data[10][:10])

# Train CBOW Word2Vec model
model = Word2Vec(
    sentences=data,
    vector_size=100,
    window=5,
    min_count=2,
    sg=0,       # 0 = CBOW, 1 = Skip-gram
    workers=1,
    epochs=100,
    seed=1
)

print("CBOW model trained successfully.")
print("Vocabulary size:", len(model.wv))

# Display word vector and similar words
if "alice" in model.wv:
    print("Vector for alice (first 10 values):", model.wv["alice"][:10])
    print("Words similar to alice:", model.wv.most_similar("alice", topn=5))

# Demonstrate context-to-target training examples
window_size = 2
contexts = []
targets = []

for sentence in data:
    for i, target in enumerate(sentence):
        context = sentence[max(0, i-window_size):i] + sentence[i+1:i+window_size+1]
        if context:
            contexts.append(context)
            targets.append(target)
        if len(targets) >= 10000:
            break
    if len(targets) >= 10000:
        break

print("Example context:", contexts[0])
print("Target word:", targets[0])

# Visualize word embeddings using PCA
words = model.wv.index_to_key[:40]
vectors = np.array([model.wv[word] for word in words])
reduced = PCA(n_components=2, random_state=1).fit_transform(vectors)

plt.figure(figsize=(12, 7))
plt.scatter(reduced[:, 0], reduced[:, 1])
for i, word in enumerate(words):
    plt.annotate(word, (reduced[i, 0], reduced[i, 1]), fontsize=8)
plt.title("CBOW Word Embeddings - PCA")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Save trained Word2Vec model
model.save("cbow_word2vec.model")
print("Model saved as cbow_word2vec.model")
